# Steam Sales Dataset Analysis (Baseline Modeling & Evaluation)

This section focuses on building and evaluating baseline machine learning models to predict game pricing based on textual and structured features derived from the Steam dataset. The primary objective is to establish a performance benchmark using relatively simple and interpretable models before moving on to more advanced techniques.

In this phase, the problem is framed as a regression task, where the goal is to estimate a game’s price from its description and associated metadata.

The baseline modeling workflow includes:

- Feature Extraction – Converting game descriptions into numerical vectors using methods like Count Vectorization
- Model Training – Applying regression algorithms such as Linear Regression and Random Forest Regressor
- Performance Evaluation – Measuring model accuracy using metrics like Mean Squared Error (MSE) and R² score
- Benchmarking – Establishing a reference point for future model improvements and comparisons

By implementing these baseline models, the analysis provides an initial understanding of how well pricing can be predicted from available data and highlights the strengths and limitations of traditional machine learning approaches in this context. This serves as a foundation for future enhancements, including more sophisticated models, feature engineering, and deep learning techniques.

## Import Libraries

In [1]:
import random
import pandas as pd
import numpy as np
from dotenv import load_dotenv
from huggingface_hub import login
from tqdm.notebook import tqdm
from pathlib import Path
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb
from sales_util.evaluator import evaluate
from sales_util.items_data import Item

In [2]:
# Load env file
load_dotenv(override=True)

True

## Load Dataset From HuggingFace

In [3]:
username = "KumudithaSilva"
dataset = f"{username}/items_llm_raw_lite"

train, val, test = Item.from_hub(dataset)

items = train + val + test

print(f"Loaded {len(items):,} items")

Loaded 22,000 items


In [4]:
# import importlib
# from sales_util import evaluator

# importlib.reload(evaluator)

### Error (Mean Error)
- It tells **how far off predictions are on average**
- The difference between the predicted value and the real value.
- Smaller error = better prediction.

---

### MSE (Mean Squared Error)
- Measures **how wrong predictions are, on average**
- It squares the errors before averaging them.
- Bigger mistakes are punished more.
- Lower MSE = better model.

---

### R² (R-squared)
- Shows **how well the model explains the data**
- Value can be:
  - **1** → perfect prediction
  - **0** → same as guessing the average
  - **negative** → very bad model (worse than guessing)

| Technique | Error   | MSE | R²        |
|----------|---------|-----|-----------|
| Random   | $20.63  | 600 | -2008.2%  |
| Linear Regression   | $3.91  | 27 | 4.7%  |
| Random Forest  | $3.81  | 27 | 5.9%  |

## Random Predictor

In [5]:
def random_pricer(item):
    return random.randrange(1,49)

In [6]:
random.seed(42)
evaluate(random_pricer, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$40 $12 $2 $45 $13 $12 $12 $15 $8 $47 $1 $39 $43 $34 $1 $28 $23 $2 $3 $6 $7 $13 $32 $35 $12 $16 $12 $45 $32 $43 $34 $25 $13 $27 $33 $9 $4 $8 $39 $27 $19 $17 $6 $13 $21 $4 $7 $24 $0 $13 $23 $38 $14 $5 $35 $28 $29 $6 $20 $3 $34 $17 $37 $25 $4 $33 $11 $36 $3 $0 $37 $12 $17 $3 $10 $6 $23 $15 $24 $40 $23 $9 $22 $22 $9 $42 $16 $35 $42 $22 $5 $29 $29 $7 $31 $46 $4 $10 $28 $24 $13 $39 $39 $33 $8 $40 $19 $16 $14 $0 $20 $21 $16 $2 $11 $37 $42 $20 $11 $36 $31 $24 $38 $28 $9 $11 $8 $15 $47 $23 $28 $16 $47 $37 $22 $34 $11 $19 $4 $5 $18 $32 $5 $11 $4 $5 $31 $10 $39 $3 $28 $3 $5 $21 $38 $29 $32 $16 $21 $2 $43 $46 $8 $32 $22 $14 $38 $20 $7 $4 $24 $8 $29 $2 $46 $37 $15 $32 $2 $31 $4 $37 $20 $38 $23 $38 $8 $8 $14 $10 $33 $24 $14 $34 $13 $30 $10 $7 $23 $18 

## Linear Regression

In [7]:
def get_item_features(item):
    return {
        "peakCCU": item.peakCCU,
        "required_age": item.required_age,
        "dlc": item.dlcCount,
        "supportWindows": int(item.supportWindows),
        "supportMac": int(item.supportMac),
        "supportLinux": int(item.supportLinux),
        "positive": item.positive,
        "negative": item.negative,
        "achievements": item.achievements,
        "recommendation": item.recommendations or 0,
        "release_year": item.release_year or 0,
        "release_month": item.release_month or 0,
        "release_day": item.release_day or 0,
        "min_estimatedOwners": item.min_estimatedOwners or 0,
        "max_estimatedOwners": item.max_estimatedOwners or 0,
        "supported_languages": item.supported_languages or 0,
        "num_developers": item.num_developers or 0,
        "num_publishers": item.num_publishers or 0,
        "num_categories": item.num_categories or 0,
        "num_genres": item.num_genres or 0,
    }

In [8]:
get_item_features(items[213])

{'peakCCU': 75,
 'required_age': 0,
 'dlc': 7,
 'supportWindows': 1,
 'supportMac': 0,
 'supportLinux': 0,
 'positive': 24924,
 'negative': 6706,
 'achievements': 25,
 'recommendation': 28157,
 'release_year': 2024,
 'release_month': 9,
 'release_day': 9,
 'min_estimatedOwners': 500000,
 'max_estimatedOwners': 1000000,
 'supported_languages': 6,
 'num_developers': 1,
 'num_publishers': 1,
 'num_categories': 11,
 'num_genres': 3}

In [9]:
def list_to_df(items):
    features = [get_item_features(item) for item in items]
    df = pd.DataFrame(features)
    df['price'] = [item.price for item in items]
    return df

In [10]:
train_df = list_to_df(train)
test_df = list_to_df(test)

In [11]:
# Linear Regression

np.random.seed(42)

feature_columns = train_df.columns.drop('price')

X_train = train_df[feature_columns]
y_train = train_df['price']
X_test = test_df[feature_columns]
y_test = test_df['price']

model = LinearRegression()
model.fit(X_train, y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


In [12]:
for feature, coef in zip(feature_columns, model.coef_):
    print(f"{feature}: {coef}")
print(f"Intercept: {model.intercept_}")

peakCCU: 0.0002717383810065009
required_age: 0.11507213336280317
dlc: 0.3702579179946405
supportWindows: 3.4845156145400877
supportMac: 0.3386431328902847
supportLinux: -0.9097494432716637
positive: 8.02274239495842e-05
negative: 0.00028868965295404317
achievements: -0.0004976915990368943
recommendation: -5.412356649351636e-06
release_year: 0.15264366392702405
release_month: 0.03772302041013199
release_day: 0.020253304463804297
min_estimatedOwners: -2.4408136708726005e-06
max_estimatedOwners: 1.0677525821195423e-06
supported_languages: -0.03172688741949436
num_developers: 0.017152079932861313
num_publishers: 0.319854496763135
num_categories: 0.18424029095022113
num_genres: 0.019054380100723377
Intercept: -308.61968636827794


In [13]:
y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

In [14]:
print(f"MSE: {mse}")
print("R²:", r2)

MSE: 27.727763760901983
R²: 0.0585138414441847


In [15]:
def linear_regression_pricer(item):
    features = get_item_features(item)
    features_df = pd.DataFrame([features])
    return model.predict(features_df)[0]

In [16]:
evaluate(linear_regression_pricer, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$5 $15 $1 $2 $2 $24 $3 $3 $4 $3 $2 $1 $1 $4 $1 $5 $0 $0 $1 $4 $2 $2 $4 $1 $9 $15 $4 $4 $4 $3 $4 $4 $2 $3 $1 $5 $0 $2 $1 $4 $1 $4 $10 $4 $3 $2 $7 $4 $3 $5 $4 $4 $3 $3 $8 $3 $1 $4 $1 $4 $3 $1 $3 $9 $13 $2 $2 $5 $3 $2 $0 $2 $11 $4 $1 $4 $3 $2 $1 $3 $4 $1 $3 $4 $1 $4 $4 $5 $4 $15 $6 $5 $7 $1 $0 $3 $15 $4 $4 $3 $0 $2 $1 $1 $2 $1 $3 $14 $4 $1 $3 $0 $3 $2 $2 $5 $0 $4 $3 $2 $4 $3 $3 $2 $5 $0 $4 $5 $4 $7 $1 $4 $3 $3 $1 $0 $9 $0 $4 $10 $8 $4 $3 $10 $5 $0 $5 $4 $2 $19 $6 $2 $8 $1 $4 $5 $3 $3 $11 $2 $4 $4 $10 $7 $8 $2 $4 $4 $2 $9 $2 $2 $3 $2 $1 $5 $4 $4 $5 $3 $2 $1 $6 $2 $5 $2 $1 $3 $4 $4 $2 $5 $10 $0 $3 $3 $8 $4 $3 $3 

## CountVector

In [17]:
prices = np.array([float(item.price) for item in train])
description = [item.small_description for item in train]

In [18]:
np.random.seed(42)
vectorize = CountVectorizer(max_features=2000, stop_words='english')
X = vectorize.fit_transform(description)

In [19]:
selected_words = vectorize.get_feature_names_out()
print(f"Number of selected words: {len(selected_words)}")
print("Selected words:", selected_words[1000:1020])

Number of selected words: 2000
Selected words: ['leaderboard' 'leaderboards' 'learn' 'learning' 'legacy' 'legend'
 'legendary' 'legends' 'letters' 'level' 'levels' 'library' 'life' 'lift'
 'light' 'like' 'limited' 'limitless' 'line' 'linear']


## Linear Regression With CountVector

In [20]:
regressor = LinearRegression()
regressor.fit(X, prices)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


In [21]:
def natural_language_linear_regression_pricer(item):
    x = vectorize.transform([item.small_description])
    return max(regressor.predict(x)[0], 0)

In [22]:
evaluate(natural_language_linear_regression_pricer, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$3 $17 $2 $3 $0 $29 $3 $3 $7 $1 $5 $1 $2 $1 $4 $8 $1 $2 $1 $4 $0 $1 $6 $1 $9 $13 $6 $2 $3 $2 $1 $2 $2 $1 $2 $1 $0 $0 $1 $4 $5 $1 $7 $6 $2 $1 $8 $2 $2 $6 $3 $3 $2 $4 $5 $1 $2 $6 $6 $4 $4 $3 $0 $9 $13 $2 $5 $5 $2 $1 $0 $1 $5 $0 $1 $1 $3 $2 $2 $5 $5 $6 $2 $4 $2 $6 $1 $2 $7 $14 $4 $9 $8 $0 $0 $1 $13 $3 $2 $4 $2 $3 $5 $2 $4 $2 $0 $10 $3 $5 $2 $2 $3 $1 $1 $4 $4 $3 $5 $1 $8 $2 $1 $5 $1 $2 $4 $0 $6 $9 $3 $0 $1 $7 $2 $1 $9 $2 $7 $4 $4 $2 $6 $12 $4 $3 $6 $3 $0 $18 $7 $4 $13 $2 $3 $6 $0 $4 $12 $2 $6 $2 $9 $9 $2 $4 $3 $4 $6 $13 $6 $3 $3 $2 $2 $6 $0 $5 $4 $4 $3 $0 $5 $1 $7 $3 $1 $1 $3 $2 $1 $1 $9 $0 $4 $3 $5 $3 $1 $4 

## Random Forest

In [23]:
rf_model = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=4)

rf_model.fit(X_train, y_train)

y_pred = rf_model.predict(X_test)

In [24]:
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

In [25]:
print("MSE:", mse)
print("R²:", r2)

MSE: 24.618547144678676
R²: 0.16408616358917005


In [26]:
def random_forest_pricer(item):
    features = get_item_features(item)
    features_df = pd.DataFrame([features])
    return rf_model.predict(features_df)[0]

In [27]:
evaluate(random_forest_pricer, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$3 $15 $1 $2 $0 $19 $5 $4 $2 $3 $3 $4 $2 $7 $0 $6 $0 $1 $1 $4 $0 $2 $1 $0 $7 $14 $8 $4 $0 $1 $4 $2 $3 $3 $0 $4 $1 $2 $1 $4 $2 $2 $5 $3 $7 $0 $2 $3 $1 $0 $2 $3 $0 $0 $6 $2 $1 $3 $1 $3 $2 $0 $1 $9 $11 $7 $3 $0 $6 $1 $2 $1 $10 $2 $1 $2 $6 $1 $3 $3 $2 $1 $1 $4 $1 $4 $2 $8 $6 $17 $5 $2 $9 $1 $4 $2 $15 $3 $1 $2 $1 $2 $3 $3 $2 $0 $1 $14 $3 $2 $4 $1 $1 $4 $3 $6 $1 $6 $1 $2 $2 $1 $0 $2 $5 $0 $3 $6 $6 $8 $2 $7 $4 $2 $12 $0 $9 $0 $3 $13 $7 $2 $3 $7 $5 $1 $6 $3 $10 $13 $3 $1 $8 $1 $3 $4 $3 $4 $8 $2 $4 $4 $10 $8 $7 $2 $4 $4 $2 $6 $0 $2 $3 $1 $4 $8 $7 $2 $6 $1 $7 $0 $6 $1 $7 $12 $1 $3 $5 $4 $4 $4 $11 $0 $1 $3 $7 $2 $3 $3 

## XGBoost

In [28]:
xgb_model = xgb.XGBRegressor(n_estimators=1000, random_state=42, n_jobs=4, learning_rate=0.1)

xgb_model.fit(X_train, y_train)

y_pred = xgb_model.predict(X_test)

In [29]:
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

In [30]:
print("MSE:", mse)
print("R²:", r2)

MSE: 26.482690494560863
R²: 0.10078985247616523


In [31]:
def xgb_pricer(item):
    features = get_item_features(item)
    features_df = pd.DataFrame([features])
    return rf_model.predict(features_df)[0]

In [32]:
evaluate(xgb_pricer, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$3 $15 $1 $2 $0 $19 $5 $4 $2 $3 $3 $4 $2 $7 $0 $6 $0 $1 $1 $4 $0 $2 $1 $0 $7 $14 $8 $4 $0 $1 $4 $2 $3 $3 $0 $4 $1 $2 $1 $4 $2 $2 $5 $3 $7 $0 $2 $3 $1 $0 $2 $3 $0 $0 $6 $2 $1 $3 $1 $3 $2 $0 $1 $9 $11 $7 $3 $0 $6 $1 $2 $1 $10 $2 $1 $2 $6 $1 $3 $3 $2 $1 $1 $4 $1 $4 $2 $8 $6 $17 $5 $2 $9 $1 $4 $2 $15 $3 $1 $2 $1 $2 $3 $3 $2 $0 $1 $14 $3 $2 $4 $1 $1 $4 $3 $6 $1 $6 $1 $2 $2 $1 $0 $2 $5 $0 $3 $6 $6 $8 $2 $7 $4 $2 $12 $0 $9 $0 $3 $13 $7 $2 $3 $7 $5 $1 $6 $3 $10 $13 $3 $1 $8 $1 $3 $4 $3 $4 $8 $2 $4 $4 $10 $8 $7 $2 $4 $4 $2 $6 $0 $2 $3 $1 $4 $8 $7 $2 $6 $1 $7 $0 $6 $1 $7 $12 $1 $3 $5 $4 $4 $4 $11 $0 $1 $3 $7 $2 $3 $3 